In [1]:
import subprocess
from io import StringIO
import pandas as pd

cmd = [
    "aws", "s3", "ls",
    "s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/",
    "--recursive"
]

result = subprocess.run(cmd, capture_output=True, text=True, check=True)

df = pd.read_csv(
    StringIO(result.stdout),
    sep=r"\s+",
    names=["date", "time", "size", "key"],
    engine="python"
)

df.head(20)


,date,time,size,key
0,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
2,2026-02-09,04:47:50,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
3,2026-02-09,04:47:53,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
4,2026-02-09,04:47:55,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
5,2026-02-09,04:47:58,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
6,2026-02-09,04:48:02,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
7,2026-02-09,04:48:00,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
8,2026-02-09,04:48:06,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
9,2026-02-09,04:48:09,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...


In [2]:
df["size_mb"] = df["size"] / 1024 / 1024
df.groupby(df["key"].str.split("/").str[0]).agg(
    files=("key", "count"),
    size_mb=("size_mb", "sum"),
)


,files,size_mb
key,,
AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,16015,1.366166e+07


In [3]:
S3_ROOT = "s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor"

print("Analysing:", S3_ROOT)


Analysing: s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor


In [4]:
import subprocess
from io import StringIO
import pandas as pd

cmd = [
    "aws", "s3", "ls",
    S3_ROOT + "/",
    "--recursive"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    check=True
)

raw = result.stdout
print(raw.splitlines()[:5])  # preview


['2026-02-09 04:47:45  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2015/201512/galsfc3_p089r078_20151213_fcm6.tif', '2026-02-09 04:47:45  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2015/201512/galsfc3_p089r078_20151213_fcm6_clr.tif', '2026-02-09 04:47:50  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201601/galsfc3_p089r078_20160114_fcm6.tif', '2026-02-09 04:47:53  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201601/galsfc3_p089r078_20160114_fcm6_clr.tif', '2026-02-09 04:47:55  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201603/galsfc3_p089r078_20160302_fcm6.tif']


In [5]:
df = pd.read_csv(
    StringIO(raw),
    sep=r"\s+",
    names=["date", "time", "size", "key"],
    engine="python"
)

# Drop folder markers (size == 0 and ending in /)
df = df[df["key"].notna()]

df.head()


,date,time,size,key
0,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
2,2026-02-09,04:47:50,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
3,2026-02-09,04:47:53,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
4,2026-02-09,04:47:55,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...


In [6]:
from pathlib import Path

df["extension"] = df["key"].apply(
    lambda x: Path(x).suffix.lower() if "." in x else "(none)"
)

df["top_prefix"] = df["key"].str.split("/").str[0]
df["second_prefix"] = df["key"].str.split("/").str[:2].str.join("/")

df["size_mb"] = df["size"] / 1024 / 1024

df.head()


,date,time,size,key,extension,top_prefix,second_prefix,size_mb
0,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,.tif,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds,821.540924
1,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,.tif,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds,821.540924
2,2026-02-09,04:47:50,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,.tif,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds,821.540924
3,2026-02-09,04:47:53,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,.tif,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds,821.540924
4,2026-02-09,04:47:55,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,.tif,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds,821.540924


In [7]:
ext_summary = (
    df.groupby("extension")
      .agg(
          files=("key", "count"),
          size_mb=("size_mb", "sum"),
      )
      .sort_values("size_mb", ascending=False)
)

ext_summary


,files,size_mb
extension,,
.tif,16050,1.370243e+07


In [8]:
tiles_df = df[df["key"].str.contains("eds/tiles/")].copy()

tiles_df["after_tiles"] = tiles_df["key"].str.split("eds/tiles/", n=1).str[1]

tiles_df[["after_tiles", "size_mb"]].head(20)


,after_tiles,size_mb
0,p089r078/fc/2015/201512/galsfc3_p089r078_20151...,821.540924
1,p089r078/fc/2015/201512/galsfc3_p089r078_20151...,821.540924
2,p089r078/fc/2016/201601/galsfc3_p089r078_20160...,821.540924
3,p089r078/fc/2016/201601/galsfc3_p089r078_20160...,821.540924
4,p089r078/fc/2016/201603/galsfc3_p089r078_20160...,821.540924
5,p089r078/fc/2016/201603/galsfc3_p089r078_20160...,821.540924
6,p089r078/fc/2016/201603/galsfc3_p089r078_20160...,821.540924
7,p089r078/fc/2016/201603/galsfc3_p089r078_20160...,821.540924
8,p089r078/fc/2016/201604/galsfc3_p089r078_20160...,821.540924
9,p089r078/fc/2016/201604/galsfc3_p089r078_20160...,821.540924


In [9]:
import subprocess
from io import StringIO
import pandas as pd

S3_ROOT = "s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor"

cmd = ["aws", "s3", "ls", S3_ROOT + "/", "--recursive"]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)

raw = res.stdout
print("Lines:", len(raw.splitlines()))
print("\n".join(raw.splitlines()[:5]))


Lines: 16166
2026-02-09 04:47:45  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2015/201512/galsfc3_p089r078_20151213_fcm6.tif
2026-02-09 04:47:45  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2015/201512/galsfc3_p089r078_20151213_fcm6_clr.tif
2026-02-09 04:47:50  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201601/galsfc3_p089r078_20160114_fcm6.tif
2026-02-09 04:47:53  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201601/galsfc3_p089r078_20160114_fcm6_clr.tif
2026-02-09 04:47:55  861448096 AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/fc/2016/201603/galsfc3_p089r078_20160302_fcm6.tif


In [10]:
df = pd.read_csv(
    StringIO(raw),
    sep=r"\s+",
    names=["date", "time", "bytes", "key"],
    engine="python"
)

df = df[df["key"].notna()].copy()
df["mb"] = df["bytes"] / 1024 / 1024

df.head()


,date,time,bytes,key,mb
0,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,821.540924
1,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,821.540924
2,2026-02-09,04:47:50,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,821.540924
3,2026-02-09,04:47:53,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,821.540924
4,2026-02-09,04:47:55,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...,821.540924


In [11]:
def prefix_at_depth(key: str, depth: int) -> str:
    parts = key.split("/")
    if len(parts) <= depth:
        return "/".join(parts)
    return "/".join(parts[:depth])

PREFIX_DEPTH = 3  # change this: 2 for broad, 3 for tile-level, 4+ for deeper
df["rel_dir"] = df["key"].apply(lambda k: prefix_at_depth(k, PREFIX_DEPTH))

summary = (
    df.groupby("rel_dir", as_index=False)
      .agg(files=("key", "count"), bytes=("bytes", "sum"))
      .sort_values("bytes", ascending=False)
)

summary["size_gb"] = summary["bytes"] / 1024 / 1024 / 1024
summary.head(30)


,rel_dir,files,bytes,size_gb
0,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles,16166,14512979916498,13516.265821


In [12]:
from pathlib import Path

df["ext"] = df["key"].apply(lambda k: (Path(k).suffix.lower() or "(none)"))

ext_summary = (
    df.groupby("ext", as_index=False)
      .agg(files=("key", "count"), bytes=("bytes", "sum"))
      .sort_values("bytes", ascending=False)
)

ext_summary["size_gb"] = ext_summary["bytes"] / 1024 / 1024 / 1024
ext_summary.head(50)


,ext,files,bytes,size_gb
0,.tif,16166,14512979916498,13516.265821


In [13]:
import subprocess
from io import StringIO
import pandas as pd

S3_BASE = "s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor"

cmd = ["aws", "s3", "ls", S3_BASE + "/", "--recursive"]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)

df = pd.read_csv(
    StringIO(res.stdout),
    sep=r"\s+",
    names=["date", "time", "bytes", "key"],
    engine="python",
)
df = df[df["key"].notna()].copy()
df["bytes"] = pd.to_numeric(df["bytes"], errors="coerce").fillna(0).astype("int64")

df.head()


,date,time,bytes,key
0,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1,2026-02-09,04:47:45,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
2,2026-02-09,04:47:50,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
3,2026-02-09,04:47:53,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
4,2026-02-09,04:47:55,861448096,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...


In [14]:
S3_ROOTS = [
    f"{S3_BASE}/eds/compat/",
    f"{S3_BASE}/eds/results/",
    f"{S3_BASE}/eds/queries/",
    f"{S3_BASE}/eds/tiles/",
]


In [15]:
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd

def _depth_str(rel_dir: str) -> int:
    rel_dir = rel_dir.strip("/")
    if rel_dir == "" or rel_dir == ".":
        return 0
    return len([p for p in rel_dir.split("/") if p])

def human_bytes(n: int) -> str:
    units = ["B","KB","MB","GB","TB","PB"]
    n = float(n)
    for u in units:
        if n < 1024 or u == units[-1]:
            return f"{n:.2f} {u}"
        n /= 1024

def scan_s3_root(
    df: pd.DataFrame,
    s3_root: str,
    max_depth: int = 3,
):
    """
    S3 analogue of scan_root() using an object listing DataFrame.

    Expects df with columns: ["key", "bytes"] where key is relative to the *bucket root*.
    s3_root is full s3://... prefix ending in /
    Returns: summary, per_dir_df, ext_df
    """
    # Convert s3://bucket/prefix/ -> key prefix "prefix/"
    if not s3_root.startswith("s3://"):
        raise ValueError("s3_root must start with s3://")

    # Strip s3://bucket/
    # Example: s3://bucket/user/eds/tiles/ -> key_prefix = "user/eds/tiles/"
    parts = s3_root[5:].split("/", 1)  # remove "s3://"
    bucket = parts[0]
    key_prefix = parts[1] if len(parts) > 1 else ""
    key_prefix = key_prefix.strip("/") + "/" if key_prefix else ""

    # Filter to objects under this prefix
    sub = df[df["key"].str.startswith(key_prefix)].copy()
    if sub.empty:
        return (
            {"root": s3_root, "exists": False, "dirs_scanned": 0, "files_scanned": 0, "total_bytes": 0},
            pd.DataFrame(columns=["root","rel_dir","depth","subdirs","files","bytes"]),
            pd.DataFrame(columns=["ext","files","bytes"]),
        )

    # relative path under this root prefix
    sub["rel_path"] = sub["key"].str[len(key_prefix):]

    # extension stats
    ext_counts = Counter()
    ext_bytes = Counter()

    # dir aggregation: we treat each object as belonging to a directory
    # dir = parent folder of the object; "" means root itself
    def parent_dir(rel_path: str) -> str:
        rel_path = rel_path.strip("/")
        if "/" not in rel_path:
            return "."  # object directly under root
        return rel_path.rsplit("/", 1)[0]

    sub["rel_dir"] = sub["rel_path"].apply(parent_dir)

    # extension compute
    def ext_of(rel_path: str) -> str:
        name = rel_path.rsplit("/", 1)[-1]
        suf = Path(name).suffix.lower()
        return suf if suf else "<no_ext>"

    sub["ext"] = sub["rel_path"].apply(ext_of)

    for e, b in zip(sub["ext"], sub["bytes"]):
        ext_counts[e] += 1
        ext_bytes[e] += int(b)

    # Build directory-level stats
    dir_stats = (
        sub.groupby("rel_dir", as_index=False)
           .agg(files=("rel_path","count"), bytes=("bytes","sum"))
    )

    # Depth + prune to max_depth
    dir_stats["depth"] = dir_stats["rel_dir"].apply(_depth_str)
    dir_stats = dir_stats[dir_stats["depth"] < max_depth].copy()

    # Subdir counts: count distinct child dirs one level below each dir (within max_depth window)
    # We build a set of directories from all objects
    all_dirs = set(sub["rel_dir"].tolist())
    all_dirs.add(".")  # include root

    def subdir_count(d: str) -> int:
        d = d.strip("/")
        prefix = "" if d in (".", "") else d + "/"
        # immediate children are those dirs with exactly one more segment
        base_depth = _depth_str(d)
        children = set()
        for cand in all_dirs:
            cand = cand.strip("/")
            if cand in (".", ""):
                continue
            if prefix and not cand.startswith(prefix):
                continue
            if not prefix and cand == "":
                continue
            if not prefix:
                # root: accept any top-level dir
                if _depth_str(cand) == 1:
                    children.add(cand.split("/")[0])
            else:
                if cand.startswith(prefix) and _depth_str(cand) == base_depth + 1:
                    children.add(cand[len(prefix):].split("/")[0])
        return len(children)

    dir_stats["subdirs"] = dir_stats["rel_dir"].apply(subdir_count)

    per_dir_df = (
        dir_stats.assign(root=s3_root)
                .loc[:, ["root","rel_dir","depth","subdirs","files","bytes"]]
                .sort_values(["depth","rel_dir"])
                .reset_index(drop=True)
    )

    ext_df = (
        pd.DataFrame({
            "ext": list(ext_counts.keys()),
            "files": list(ext_counts.values()),
            "bytes": [ext_bytes[e] for e in ext_counts.keys()],
        })
        .sort_values(["bytes","files"], ascending=False)
        .reset_index(drop=True)
    )

    summary = {
        "root": s3_root,
        "exists": True,
        "dirs_scanned": int(per_dir_df.shape[0]),
        "files_scanned": int(sub.shape[0]),
        "total_bytes": int(sub["bytes"].sum()),
    }

    return summary, per_dir_df, ext_df


In [16]:
MAX_DEPTH = 7  # like your local notebook

all_summaries = []
all_per_dir = []
all_ext = []

for r in S3_ROOTS:
    summary, per_dir_df, ext_df = scan_s3_root(df, r, max_depth=MAX_DEPTH)
    all_summaries.append(summary)

    if not per_dir_df.empty:
        all_per_dir.append(per_dir_df)
    if not ext_df.empty:
        ext_df = ext_df.copy()
        ext_df["root"] = r
        all_ext.append(ext_df)

summaries_df = pd.DataFrame(all_summaries)
per_dir_all = pd.concat(all_per_dir, ignore_index=True) if all_per_dir else pd.DataFrame()
ext_all = pd.concat(all_ext, ignore_index=True) if all_ext else pd.DataFrame()

summaries_df


,root,exists,dirs_scanned,files_scanned,total_bytes
0,s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7...,False,0,0,0
1,s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7...,False,0,0,0
2,s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7...,False,0,0,0
3,s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7...,True,5596,16184,14536238508660


In [17]:
if not per_dir_all.empty:
    per_dir_all["size_gb"] = per_dir_all["bytes"] / 1024 / 1024 / 1024
    (
        per_dir_all.sort_values("bytes", ascending=False)
                  .head(50)
    )


In [18]:
if not ext_all.empty:
    ext_all2 = (
        ext_all.groupby("ext", as_index=False)
               .agg(files=("files","sum"), bytes=("bytes","sum"))
               .sort_values("bytes", ascending=False)
    )
    ext_all2["size_gb"] = ext_all2["bytes"] / 1024 / 1024 / 1024
    ext_all2


In [19]:
if not ext_all.empty:
    ext_by_root = (
        ext_all.groupby(["root","ext"], as_index=False)
               .agg(files=("files","sum"), bytes=("bytes","sum"))
               .sort_values(["root","bytes"], ascending=[True, False])
    )
    ext_by_root["size_gb"] = ext_by_root["bytes"] / 1024 / 1024 / 1024
    ext_by_root


In [20]:
tiles_root = f"{S3_BASE}/eds/tiles/"
tiles_prefix = tiles_root[5:].split("/", 1)[1].strip("/") + "/"

tiles = df[df["key"].str.startswith(tiles_prefix)].copy()
tiles["after_tiles"] = tiles["key"].str[len(tiles_prefix):]
tiles["tile_prefix"] = tiles["after_tiles"].str.split("/").str[0]

tile_summary = (
    tiles.groupby("tile_prefix", as_index=False)
         .agg(files=("after_tiles","count"), bytes=("bytes","sum"))
         .sort_values("bytes", ascending=False)
)
tile_summary["size_gb"] = tile_summary["bytes"] / 1024 / 1024 / 1024
tile_summary.head(50)


,tile_prefix,files,bytes,size_gb
29,p095r081,1180,1066227703257,993.001930
28,p095r080,1170,1059299370552,986.549417
22,p094r084,990,915330519461,852.467976
21,p094r073,980,841755584734,783.945978
27,p095r079,925,838270439996,780.700184
23,p094r085,890,820939985162,764.559941
18,p093r085,885,816648160607,760.562867
16,p093r083,799,720852280892,671.346002
30,p095r082,765,689927778246,642.545315
17,p093r084,755,689245136998,641.909556
